# 02 - Transformations SIRENE : nettoyage et enrichissement

**Projet :** Formation Data Engineer, Mois 3 Semaine 2 - PySpark & Delta Lake  
**Stack :** PySpark, Databricks Free Edition (Azure), Unity Catalog Volumes  
**Objectif :** Nettoyer les données brutes (logique `clean_nd`) et appliquer les filtres RGPD

---
| Étape | Détail |
|---|---|
| Renommage | 24 colonnes utiles (noms français → noms courts) |
| `clean_nd` | Remplacement `[ND]` et `""` → `NULL` (when/otherwise) |
| Filtre RGPD | `etat_admin_etab = "Actif"`, `statut_diffusion != "P"` (Art.25) |
| Colonnes dérivées | `loaded_at`, `siret_calcule` (lpad siren+nic), `est_siege` (boolean) |
| **Résultat** | **134 661 lignes, 27 colonnes** (établissements actifs Loire-Atlantique) |

In [0]:
from pyspark.sql import functions as F

SEP = ";"

df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", SEP)
    .option("encoding", "UTF-8")
    .csv("/Volumes/workspace/default/raw_data/sirene/data.csv")
)

print(f"Nombre de colonnes : {len(df_raw.columns)}")
print(f"Nombre de lignes : {df_raw.count()}")

Nombre de colonnes : 107
Nombre de lignes : 420411


In [0]:
rename_map = {
    "SIREN": "siren",
    "NIC": "nic",
    "SIRET": "siret",
    "Statut de diffusion de l'établissement": "statut_diffusion",
    "Date de création de l'établissement": "date_creation_etab",
    "Tranche de l'effectif de l'établissement": "tranche_effectif",
    "Activité principale de l'établissement8": "activite_principale_etab",
    "Etablissement siège": "etablissement_siege",
    "Code postal de l'établissement": "code_postal",
    "Commune de l'établissement": "commune",
    "Code commune de l'établissement": "code_commune",
    "Code du département de l'établissement": "code_departement",
    "Département de l'établissement": "departement",
    "Code de la région de l'établissement": "code_region",
    "Région de l'établissement": "region",
    "Etat administratif de l'établissement": "etat_admin_etab",
    "Date de fermeture de l'établissement": "date_fermeture_etab",
    "Dénomination de l'unité légale": "denomination_unite_legale",
    "Catégorie de l'entreprise": "categorie_entreprise",
    "Etat administratif de l'unité légale": "etat_admin_ul",
    "Caractère employeur de l'unité légale": "caractere_employeur",
    "Activité principale de l'unité légale": "activite_principale_ul",
    "Catégorie juridique de l'unité légale": "categorie_juridique",
    "Date de création de l'unité légale": "date_creation_ul",
}

# Appliquer le renommage avec garde-fou : signale toute colonne introuvable
df_renamed = df_raw
for old_name, new_name in rename_map.items():
    if old_name in df_renamed.columns:
        df_renamed = df_renamed.withColumnRenamed(old_name, new_name)
    else:
        print(f"Colonne introuvable : {old_name}")

df_renamed.select(list(rename_map.values())).printSchema()

root
 |-- siren: integer (nullable = true)
 |-- nic: integer (nullable = true)
 |-- siret: long (nullable = true)
 |-- statut_diffusion: string (nullable = true)
 |-- date_creation_etab: string (nullable = true)
 |-- tranche_effectif: string (nullable = true)
 |-- activite_principale_etab: string (nullable = true)
 |-- etablissement_siege: string (nullable = true)
 |-- code_postal: string (nullable = true)
 |-- commune: string (nullable = true)
 |-- code_commune: integer (nullable = true)
 |-- code_departement: integer (nullable = true)
 |-- departement: string (nullable = true)
 |-- code_region: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- etat_admin_etab: string (nullable = true)
 |-- date_fermeture_etab: date (nullable = true)
 |-- denomination_unite_legale: string (nullable = true)
 |-- categorie_entreprise: string (nullable = true)
 |-- etat_admin_ul: string (nullable = true)
 |-- caractere_employeur: string (nullable = true)
 |-- activite_principale_ul: st

In [0]:
df_renamed.select("etat_admin_etab").distinct().show()
df_renamed.select("statut_diffusion").distinct().show()
df_renamed.select("etablissement_siege").distinct().show()

+---------------+
|etat_admin_etab|
+---------------+
|          Fermé|
|          Actif|
+---------------+

+----------------+
|statut_diffusion|
+----------------+
|               P|
|               O|
+----------------+

+-------------------+
|etablissement_siege|
+-------------------+
|                oui|
|                non|
+-------------------+



In [0]:
colonnes_a_nettoyer = [
    "denomination_unite_legale",
    "activite_principale_etab",
    "categorie_juridique",
    "categorie_entreprise",
]

df_clean_nd = df_renamed
for c in colonnes_a_nettoyer:
    df_clean_nd = df_clean_nd.withColumn(
        c,
        F.when((F.col(c) == "[ND]") | (F.col(c) == ""), None).otherwise(F.col(c))
    )

In [0]:
df_filtered = (
    df_clean_nd
    .filter(F.col("etat_admin_etab") == "Actif")
    .filter(F.col("statut_diffusion") != "P")
)

print(f"Lignes avant filtre : {df_clean_nd.count()}")
print(f"Lignes après filtre : {df_filtered.count()}")

Lignes avant filtre : 420411
Lignes après filtre : 134661


In [0]:
df_enriched = (
    df_filtered
    .withColumn("loaded_at", F.current_timestamp())
    .withColumn("siret_calcule", F.concat(F.col("siren"), F.col("nic")))
    .withColumn(
        "est_siege",
        F.when(F.col("etablissement_siege").isin("true", "oui", "True", "1"), True)
         .when(F.col("etablissement_siege").isin("false", "non", "False", "0"), False)
         .otherwise(None)
    )
)

df_enriched.select("siren", "siret_calcule", "est_siege", "loaded_at").show(5, truncate=False)

+---------+-------------+---------+--------------------------+
|siren    |siret_calcule|est_siege|loaded_at                 |
+---------+-------------+---------+--------------------------+
|808719801|80871980133  |true     |2026-07-09 10:36:48.042491|
|920181153|92018115315  |true     |2026-07-09 10:36:48.042491|
|892492018|89249201815  |true     |2026-07-09 10:36:48.042491|
|889847356|88984735617  |true     |2026-07-09 10:36:48.042491|
|501700744|50170074437  |true     |2026-07-09 10:36:48.042491|
+---------+-------------+---------+--------------------------+
only showing top 5 rows


In [0]:
# Répartition actif/fermé
df_enriched.groupBy("etat_admin_etab").count().show()

# Validation département - doit être très majoritairement 44
df_enriched.groupBy("code_departement").count().orderBy(F.desc("count")).show()

# Top 10 communes
df_enriched.groupBy("commune").count().orderBy(F.desc("count")).show(10, truncate=False)

+---------------+------+
|etat_admin_etab| count|
+---------------+------+
|          Actif|134661|
+---------------+------+

+----------------+------+
|code_departement| count|
+----------------+------+
|              44|134661|
+----------------+------+

+-------------------------+-----+
|commune                  |count|
+-------------------------+-----+
|NANTES                   |74724|
|SAINT-HERBLAIN           |10268|
|REZE                     |7159 |
|VERTOU                   |4819 |
|ORVAULT                  |4742 |
|CARQUEFOU                |4253 |
|SAINT-SEBASTIEN-SUR-LOIRE|3674 |
|BOUGUENAIS               |3258 |
|LA CHAPELLE-SUR-ERDRE    |3098 |
|COUERON                  |3040 |
+-------------------------+-----+
only showing top 10 rows


In [0]:
df_clean = df_enriched

print(f"Nombre de colonnes final : {len(df_clean.columns)}")
print(f"Nombre de lignes final : {df_clean.count()}")
df_clean.printSchema()

Nombre de colonnes final : 110
Nombre de lignes final : 134661
root
 |-- siren: integer (nullable = true)
 |-- nic: integer (nullable = true)
 |-- siret: long (nullable = true)
 |-- statut_diffusion: string (nullable = true)
 |-- date_creation_etab: string (nullable = true)
 |-- tranche_effectif: string (nullable = true)
 |-- Tranche de l'effectif de l'établissement triable: integer (nullable = true)
 |-- Année de la tranche d'effectif de l'établissement: integer (nullable = true)
 |-- activite_principale_etab: string (nullable = true)
 |-- Date de la dernière mise à jour de l'établissement: timestamp (nullable = true)
 |-- etablissement_siege: string (nullable = true)
 |-- Nombre de periodes de l'établissement: integer (nullable = true)
 |-- Complément d'adresse de l'établissement: string (nullable = true)
 |-- Numéro de voie de l'établissement: integer (nullable = true)
 |-- Indice de répétition de l'établissement: string (nullable = true)
 |-- Type de voie de l'établissement: string